# Phase 4.3 — Hybrid NER / Information Extraction

**Disaster Intelligence Platform**

This notebook implements the Phase 4.3 hybrid information-extraction pipeline:

`HumAID tweet → general NER → disaster-specific rules/patterns/gazetteers → normalization → merging/conflict resolution → structured incident JSON → gold-standard evaluation`

### Design decision
We are using a **hybrid** approach rather than treating a general-purpose NER model as a complete disaster-information extractor.

The general NER model provides `PERSON`, `ORGANIZATION`, and `LOCATION`-type entities. Disaster-specific rules then target `CASUALTY`, `DISPLACED`, `REQUEST`, `RESOURCE`, `RESCUE`, `DISASTER_TYPE`, `NUMBER`, and `INFRASTRUCTURE`.

> Important: rule-based outputs are **candidate extractions**, not automatically ground truth. Final quality is measured against a small manually annotated gold-standard sample.


## 0. Project conventions

We follow the lessons from Phase 3:

- Do not assume dataset column names.
- Do not assume a `dev` split.
- Do not depend on `tweet_id`.
- Make the notebook runnable top-to-bottom.
- Keep intermediate outputs inspectable.
- Pin the same Transformers/tokenizers versions used in the project.
- Save structured outputs at the end.

The general NER model used here is `dslim/bert-base-NER`, an English BERT model trained on CoNLL-2003. It recognizes `PER`, `ORG`, `LOC`, and `MISC`; therefore, the disaster-specific layer is necessary for the project's humanitarian entities. citeturn0search1turn0search2


In [ ]:
# ============================================================
# 1. ENVIRONMENT
# ============================================================

!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "datasets>=2.20,<3"     "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3" "tqdm>=4.65"

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import precision_recall_fscore_support

import torch
from transformers import pipeline

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python environment ready.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Configuration

**Change only the dataset-loading configuration if your HumAID files are stored locally.**

The loader below supports:

1. Hugging Face `datasets` loading, or
2. local CSV/CSV.GZ/Parquet files.

It deliberately inspects actual columns and split names rather than assuming `train/dev/test`.


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/phase4_3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If your Kaggle environment can load HumAID directly, keep this True.
USE_HF_DATASET = True

# Your project uses QCRI/HumAID-all.
HF_DATASET_ID = "QCRI/HumAID-all"

# If you use local files instead, set this to the folder containing
# train/validation/test CSV or Parquet files.
LOCAL_DATA_DIR = Path("/kaggle/input/humaid")

# General-purpose English NER model.
NER_MODEL_NAME = "dslim/bert-base-NER"

# Maximum number of tweets used for quick development runs.
# Set to None for the complete selected split.
DEMO_LIMIT = 2000

# Minimum confidence for general NER entities.
GENERAL_NER_THRESHOLD = 0.55

# Rule confidence levels are intentionally conservative.
RULE_HIGH = 0.92
RULE_MEDIUM = 0.85
RULE_LOW = 0.75

print("Output directory:", OUTPUT_DIR)
print("General NER model:", NER_MODEL_NAME)


In [ ]:
# ============================================================
# 3. LOAD HUM AID — ROBUSTLY
# ============================================================

from datasets import load_dataset, DatasetDict

def inspect_dataset(ds):
    print("Dataset type:", type(ds))
    if hasattr(ds, "keys"):
        print("Splits:", list(ds.keys()))
        for split_name in ds.keys():
            split = ds[split_name]
            print(f"  {split_name}: {len(split):,} rows")
            print("    columns:", split.column_names)
    else:
        print("Columns:", ds.column_names)

def find_text_column(columns):
    preferred = [
        "text", "tweet", "tweet_text", "content", "post",
        "message", "tweetText", "clean_text"
    ]
    lower_map = {str(c).lower(): c for c in columns}

    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    # Conservative fallback: choose a string-like name containing text/tweet.
    candidates = [
        c for c in columns
        if any(k in str(c).lower() for k in ["text", "tweet", "content", "message", "post"])
    ]
    if candidates:
        return candidates[0]

    raise ValueError(
        f"Could not identify a text column. Actual columns are: {list(columns)}"
    )

if USE_HF_DATASET:
    # HumAID's dataset script declares its validation split as "dev" in
    # metadata but actually generates "validation". Without
    # verification_mode="no_checks" this raises ExpectedMoreSplitsError.
    dataset = load_dataset(HF_DATASET_ID, verification_mode="no_checks")
else:
    if not LOCAL_DATA_DIR.exists():
        raise FileNotFoundError(
            f"Local data directory does not exist: {LOCAL_DATA_DIR}"
        )

    files = sorted(
        list(LOCAL_DATA_DIR.glob("*.csv")) +
        list(LOCAL_DATA_DIR.glob("*.csv.gz")) +
        list(LOCAL_DATA_DIR.glob("*.parquet"))
    )

    if not files:
        raise FileNotFoundError(
            f"No CSV/CSV.GZ/Parquet files found in {LOCAL_DATA_DIR}"
        )

    loaded = {}
    for file in files:
        name = file.stem.replace(".csv", "")
        if file.suffix == ".parquet":
            df = pd.read_parquet(file)
        else:
            df = pd.read_csv(file)
        loaded[name] = df

    dataset = DatasetDict({
        name: __import__("datasets").Dataset.from_pandas(df, preserve_index=False)
        for name, df in loaded.items()
    })

inspect_dataset(dataset)

# Determine the actual split to use.
split_names = list(dataset.keys())

preferred_splits = ["validation", "dev", "test", "train"]
SELECTED_SPLIT = next(
    (s for s in preferred_splits if s in split_names),
    split_names[0]
)

TEXT_COLUMN = find_text_column(dataset[SELECTED_SPLIT].column_names)

print("\nSelected split:", SELECTED_SPLIT)
print("Detected text column:", TEXT_COLUMN)


In [ ]:
# ============================================================
# 4. CONVERT TO DATAFRAME + BASIC SAFETY CHECKS
# ============================================================

df = dataset[SELECTED_SPLIT].to_pandas().copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df = df[df[TEXT_COLUMN].str.strip().ne("")].copy()
df = df.reset_index(drop=True)

if DEMO_LIMIT is not None:
    df = df.head(DEMO_LIMIT).copy()

print("Rows selected:", len(df))
print("Columns:", list(df.columns))

display(df[[TEXT_COLUMN]].head(10))


## 5. Load the general-purpose NER model

We use `dslim/bert-base-NER` as the general NER layer. Its documented entity types are `PER`, `ORG`, `LOC`, and `MISC`. The Transformers token-classification pipeline supports entity aggregation, so we use `aggregation_strategy="simple"` to combine token-level predictions into entity spans. citeturn0search2turn0search3

This model is **not** expected to solve disaster-specific entities such as casualties or resource requests by itself.


In [ ]:
# ============================================================
# 5. GENERAL NER
# ============================================================

device = 0 if torch.cuda.is_available() else -1

general_ner = pipeline(
    "token-classification",
    model=NER_MODEL_NAME,
    tokenizer=NER_MODEL_NAME,
    aggregation_strategy="simple",
    device=device
)

print("General NER pipeline loaded.")


In [ ]:
# ============================================================
# 6. TEST GENERAL NER
# ============================================================

TEST_TEXTS = [
    "Flooding in Mumbai has damaged several roads.",
    "Red Cross volunteers are distributing food in Houston.",
    "John was rescued from a collapsed building in Manila.",
    "People in New Orleans need drinking water."
]

for text in TEST_TEXTS:
    print("\nTEXT:", text)
    for entity in general_ner(text):
        print(entity)


In [ ]:
# ============================================================
# 7. ENTITY SCHEMA
# ============================================================

TARGET_ENTITY_TYPES = [
    "LOCATION",
    "CASUALTY",
    "DISPLACED",
    "REQUEST",
    "RESOURCE",
    "RESCUE",
    "DISASTER_TYPE",
    "ORGANIZATION",
    "PERSON",
    "NUMBER",
    "INFRASTRUCTURE"
]

GENERAL_LABEL_MAP = {
    "LOC": "LOCATION",
    "PER": "PERSON",
    "ORG": "ORGANIZATION",
    "MISC": "MISC"
}

print("Target schema:")
for label in TARGET_ENTITY_TYPES:
    print(" -", label)


# 8. Disaster-specific gazetteers

Gazetteers are small curated vocabularies used as one signal in the extraction layer.

They are deliberately separated from rules so that they can later be expanded from domain knowledge, historical data, or manually reviewed false negatives.


In [ ]:
# ============================================================
# 8. GAZETTEERS
# ============================================================

DISASTER_TYPES = {
    "flood": [
        "flood", "flooding", "flash flood", "flash flooding",
        "floodwaters", "flood water", "inundation"
    ],
    "earthquake": [
        "earthquake", "quake", "aftershock", "tremor"
    ],
    "fire": [
        "fire", "wildfire", "forest fire", "bushfire", "blaze"
    ],
    "hurricane": [
        "hurricane", "cyclone", "typhoon", "tropical storm"
    ],
    "tornado": [
        "tornado", "twister"
    ],
    "landslide": [
        "landslide", "mudslide", "rockslide"
    ],
    "tsunami": [
        "tsunami"
    ],
    "volcano": [
        "volcanic eruption", "volcano", "eruption"
    ],
    "storm": [
        "storm", "thunderstorm", "windstorm"
    ]
}

RESOURCE_TERMS = [
    "water", "drinking water", "food", "meals", "blankets",
    "medicine", "medicines", "medical supplies", "supplies",
    "clothes", "shelter", "tents", "fuel", "baby food",
    "bottled water", "first aid", "first aid kits",
    "blood", "oxygen", "generators", "generators"
]

INFRASTRUCTURE_TERMS = [
    "bridge", "road", "highway", "street", "building",
    "house", "homes", "hospital", "school", "airport",
    "railway", "railroad", "station", "dam", "power line",
    "power lines", "electricity grid", "pipeline", "port",
    "roadway", "overpass", "underpass"
]

RESCUE_TERMS = [
    "rescue", "rescued", "rescuing", "trapped", "stranded",
    "stuck", "evacuate", "evacuated", "evacuation",
    "missing", "search and rescue", "sos", "save us"
]

REQUEST_TERMS = [
    "need", "needs", "needed", "require", "requires",
    "required", "request", "requesting", "please send",
    "please provide", "looking for", "asking for",
    "urgently need", "urgent need", "help needed",
    "help us"
]

DISPLACEMENT_TERMS = [
    "displaced", "homeless", "evacuated", "evacuation",
    "forced to leave", "left their homes", "lost their homes",
    "without shelter", "shelter needed"
]

print("Gazetteers loaded.")
print("Disaster types:", len(DISASTER_TYPES))
print("Resources:", len(RESOURCE_TERMS))
print("Infrastructure:", len(INFRASTRUCTURE_TERMS))


In [ ]:
# ============================================================
# 9. REGEX PATTERNS
# ============================================================

NUMBER_RE = re.compile(
    r"(?<!\w)(\d{1,6}(?:[,.]\d{3})*(?:\.\d+)?)(?!\w)",
    re.IGNORECASE
)

CASUALTY_RE = re.compile(
    r"(?P<number>\d{1,6})\s*"
    r"(?P<descriptor>people|persons|person|residents|victims|"
    r"children|men|women|families|workers)?\s*"
    r"(?:are|were|have\s+been|had\s+been)?\s*"
    r"(?P<status>killed|dead|died|deadly|injured|hurt|missing|"
    r"trapped|rescued|fatalities|casualties|victims)",
    re.IGNORECASE
)

CASUALTY_REVERSE = re.compile(
    r"(?P<status>\d{1,6})?\s*"
    r"(?P<descriptor>fatalities|casualties|deaths|injuries|victims)",
    re.IGNORECASE
)

REQUEST_RE = re.compile(
    r"(?P<context>"
    r"(?:urgently\s+)?(?:need|needs|needed|require|requires|required|"
    r"request|requesting|please\s+(?:send|provide)|help\s+(?:needed|us)|"
    r"looking\s+for|asking\s+for)"
    r")\s+"
    r"(?P<item>[^.!?;,\n]{2,80})",
    re.IGNORECASE
)

DISPLACED_RE = re.compile(
    r"(?P<text>"
    r"\d{0,6}\s*(?:people|families|residents)?\s*"
    r"(?:are\s+)?(?:displaced|homeless|evacuated)|"
    r"(?:people|families|residents)\s+(?:were|are)\s+"
    r"(?:forced\s+to\s+leave|displaced)|"
    r"(?:lost|have\s+lost)\s+(?:their\s+)?homes"
    r")",
    re.IGNORECASE
)

print("Regex patterns compiled.")


In [ ]:
# ============================================================
# 10. UTILITY FUNCTIONS
# ============================================================

def clean_span(text):
    return re.sub(r"\s+", " ", str(text)).strip(" ,.;:!?")

def add_entity(entities, text, start, end, label, score=1.0, source="rule"):
    text = clean_span(text)
    if not text or start is None or end is None or end <= start:
        return

    entities.append({
        "text": text,
        "start": int(start),
        "end": int(end),
        "label": label,
        "score": float(score),
        "source": source
    })

def overlaps(a, b):
    return max(a["start"], b["start"]) < min(a["end"], b["end"])

def normalize_label(label):
    return {
        "PER": "PERSON",
        "ORG": "ORGANIZATION",
        "LOC": "LOCATION",
        "MISC": "MISC"
    }.get(label, label)

def normalize_entity_text(text, label):
    text = clean_span(text)

    if label in {"LOCATION", "ORGANIZATION", "PERSON", "INFRASTRUCTURE"}:
        return text.strip()

    return text.lower()

def deduplicate_entities(entities):
    # Prefer higher confidence when exact span + label duplicates occur.
    best = {}

    for e in entities:
        key = (
            e["start"],
            e["end"],
            e["label"],
            normalize_entity_text(e["text"], e["label"])
        )

        if key not in best or e["score"] > best[key]["score"]:
            best[key] = e

    return list(best.values())

def merge_adjacent_same_label(entities, text, max_gap=2):
    entities = sorted(entities, key=lambda x: (x["start"], x["end"]))
    merged = []

    for e in entities:
        if not merged:
            merged.append(e.copy())
            continue

        prev = merged[-1]
        gap = text[prev["end"]:e["start"]]

        if (
            prev["label"] == e["label"]
            and len(gap) <= max_gap
            and gap.strip() == ""
        ):
            prev["end"] = e["end"]
            prev["text"] = text[prev["start"]:prev["end"]]
            prev["score"] = max(prev["score"], e["score"])
            prev["source"] = prev["source"] + "+" + e["source"]
        else:
            merged.append(e.copy())

    return merged


# 11. General NER extraction

The model's output is normalized into the project's schema:

- `PER → PERSON`
- `ORG → ORGANIZATION`
- `LOC → LOCATION`

`MISC` is retained internally but is not automatically forced into one of the humanitarian classes because that would create unjustified labels.


In [ ]:
# ============================================================
# 11. GENERAL NER EXTRACTION
# ============================================================

def extract_general_ner(text):
    entities = []

    if not text.strip():
        return entities

    predictions = general_ner(text)

    for p in predictions:
        score = float(p.get("score", 0.0))
        raw_label = p.get("entity_group", p.get("entity", ""))

        if score < GENERAL_NER_THRESHOLD:
            continue

        label = normalize_label(raw_label)

        if label not in {"LOCATION", "ORGANIZATION", "PERSON", "MISC"}:
            continue

        start = p.get("start")
        end = p.get("end")

        if start is None or end is None:
            # Fallback only when exact offsets are unavailable.
            word = clean_span(p.get("word", ""))
            start = text.find(word)
            end = start + len(word) if start >= 0 else None

        if start is not None and start >= 0 and end is not None:
            add_entity(
                entities,
                text[start:end],
                start,
                end,
                label,
                score,
                source="general_ner"
            )

    return entities


# 12. Disaster-specific extraction

This layer is intentionally transparent and inspectable.

Each rule records:

- extracted text
- character offsets
- target label
- confidence
- extraction source

This will later make error analysis much easier.


In [ ]:
# ============================================================
# 12A. DISASTER TYPE
# ============================================================

def extract_disaster_type(text):
    entities = []
    lower = text.lower()

    for canonical, terms in DISASTER_TYPES.items():
        for term in terms:
            for match in re.finditer(
                rf"(?<!\w){re.escape(term)}(?!\w)",
                lower,
                flags=re.IGNORECASE
            ):
                add_entity(
                    entities,
                    text[match.start():match.end()],
                    match.start(),
                    match.end(),
                    "DISASTER_TYPE",
                    RULE_HIGH,
                    source="disaster_gazetteer"
                )

    return entities


# ============================================================
# 12B. NUMBERS
# ============================================================

def extract_numbers(text):
    entities = []

    for match in NUMBER_RE.finditer(text):
        add_entity(
            entities,
            match.group(1),
            match.start(1),
            match.end(1),
            "NUMBER",
            RULE_HIGH,
            source="number_regex"
        )

    return entities


# ============================================================
# 12C. CASUALTIES
# ============================================================

def extract_casualties(text):
    entities = []

    for match in CASUALTY_RE.finditer(text):
        start = match.start()
        end = match.end()

        add_entity(
            entities,
            text[start:end],
            start,
            end,
            "CASUALTY",
            RULE_HIGH,
            source="casualty_regex"
        )

    # Handle expressions such as "5 fatalities".
    for match in re.finditer(
        r"(?<!\w)(\d{1,6})\s+"
        r"(fatalities|casualties|deaths|injuries|victims)(?!\w)",
        text,
        flags=re.IGNORECASE
    ):
        add_entity(
            entities,
            match.group(0),
            match.start(),
            match.end(),
            "CASUALTY",
            RULE_HIGH,
            source="casualty_regex"
        )

    return entities


# ============================================================
# 12D. REQUESTS
# ============================================================

def extract_requests(text):
    entities = []

    for match in REQUEST_RE.finditer(text):
        item = clean_span(match.group("item"))
        item_start = match.start("item")
        item_end = match.end("item")

        if len(item) > 80:
            item = item[:80].rstrip()

        add_entity(
            entities,
            item,
            item_start,
            min(item_end, item_start + len(item)),
            "REQUEST",
            RULE_MEDIUM,
            source="request_regex"
        )

    return entities


# ============================================================
# 12E. RESOURCES
# ============================================================

def extract_resources(text):
    entities = []
    lower = text.lower()

    for term in sorted(RESOURCE_TERMS, key=len, reverse=True):
        for match in re.finditer(
            rf"(?<!\w){re.escape(term)}(?!\w)",
            lower
        ):
            add_entity(
                entities,
                text[match.start():match.end()],
                match.start(),
                match.end(),
                "RESOURCE",
                RULE_MEDIUM,
                source="resource_gazetteer"
            )

    return entities


# ============================================================
# 12F. RESCUE
# ============================================================

def extract_rescue(text):
    entities = []
    lower = text.lower()

    for term in sorted(RESCUE_TERMS, key=len, reverse=True):
        for match in re.finditer(
            rf"(?<!\w){re.escape(term)}(?!\w)",
            lower
        ):
            add_entity(
                entities,
                text[match.start():match.end()],
                match.start(),
                match.end(),
                "RESCUE",
                RULE_MEDIUM,
                source="rescue_gazetteer"
            )

    return entities


# ============================================================
# 12G. DISPLACEMENT
# ============================================================

def extract_displaced(text):
    entities = []

    for match in DISPLACED_RE.finditer(text):
        add_entity(
            entities,
            match.group(0),
            match.start(),
            match.end(),
            "DISPLACED",
            RULE_MEDIUM,
            source="displacement_regex"
        )

    lower = text.lower()

    for term in DISPLACEMENT_TERMS:
        for match in re.finditer(
            rf"(?<!\w){re.escape(term)}(?!\w)",
            lower
        ):
            add_entity(
                entities,
                text[match.start():match.end()],
                match.start(),
                match.end(),
                "DISPLACED",
                RULE_LOW,
                source="displacement_gazetteer"
            )

    return entities


# ============================================================
# 12H. INFRASTRUCTURE
# ============================================================

def extract_infrastructure(text):
    entities = []
    lower = text.lower()

    for term in sorted(INFRASTRUCTURE_TERMS, key=len, reverse=True):
        for match in re.finditer(
            rf"(?<!\w){re.escape(term)}(?!\w)",
            lower
        ):
            add_entity(
                entities,
                text[match.start():match.end()],
                match.start(),
                match.end(),
                "INFRASTRUCTURE",
                RULE_MEDIUM,
                source="infrastructure_gazetteer"
            )

    return entities


In [ ]:
# ============================================================
# 13. CONTEXT-AWARE REQUEST + RESOURCE LINKING
# ============================================================

def link_request_to_resources(text, request_entities, resource_entities):
    """
    Add a lightweight context relation without changing entity labels.

    Example:
        "Need drinking water and blankets"
    gives:
        REQUEST -> "drinking water and blankets"
        RESOURCE -> "drinking water"
        RESOURCE -> "blankets"

    The relation is represented later in the structured JSON.
    """
    links = []

    for req in request_entities:
        nearby = [
            r for r in resource_entities
            if abs(r["start"] - req["start"]) <= 120
        ]

        for r in nearby:
            links.append({
                "request_text": req["text"],
                "resource_text": r["text"],
                "request_span": [req["start"], req["end"]],
                "resource_span": [r["start"], r["end"]]
            })

    return links


# 14. Entity conflict resolution

Overlaps are expected.

For example, a casualty phrase may contain a `NUMBER` entity:

`20 people injured`

We want the higher-level `CASUALTY` span to remain while the `NUMBER` remains available as a separate numerical signal.

Therefore, conflict resolution is **label-aware**, not simply "keep the first entity".


In [ ]:
# ============================================================
# 14. CONFLICT RESOLUTION
# ============================================================

PARENT_ENTITY_PRIORITY = {
    "CASUALTY": 100,
    "DISPLACED": 90,
    "REQUEST": 80,
    "RESCUE": 80,
    "DISASTER_TYPE": 70,
    "INFRASTRUCTURE": 60,
    "RESOURCE": 50,
    "LOCATION": 40,
    "ORGANIZATION": 40,
    "PERSON": 40,
    "NUMBER": 10,
    "MISC": 1
}

def resolve_overlaps(entities):
    entities = deduplicate_entities(entities)

    # Sort by priority first, then confidence and span length.
    ordered = sorted(
        entities,
        key=lambda e: (
            PARENT_ENTITY_PRIORITY.get(e["label"], 0),
            e["score"],
            e["end"] - e["start"]
        ),
        reverse=True
    )

    kept = []

    for candidate in ordered:
        conflicting = False

        for existing in kept:
            if overlaps(candidate, existing):
                # Keep both when the smaller entity is NUMBER and the
                # larger entity is a humanitarian phrase.
                if (
                    candidate["label"] == "NUMBER"
                    and existing["label"] != "NUMBER"
                ) or (
                    existing["label"] == "NUMBER"
                    and candidate["label"] != "NUMBER"
                ):
                    continue

                conflicting = True
                break

        if not conflicting:
            kept.append(candidate)

    return sorted(kept, key=lambda e: (e["start"], e["end"]))


In [ ]:
# ============================================================
# 15. FULL HYBRID EXTRACTION FOR ONE TWEET
# ============================================================

def extract_hybrid(text):
    text = str(text)

    entities = []

    # General NER
    entities.extend(extract_general_ner(text))

    # Disaster-specific layer
    entities.extend(extract_disaster_type(text))
    entities.extend(extract_numbers(text))
    entities.extend(extract_casualties(text))
    entities.extend(extract_requests(text))
    entities.extend(extract_resources(text))
    entities.extend(extract_rescue(text))
    entities.extend(extract_displaced(text))
    entities.extend(extract_infrastructure(text))

    entities = deduplicate_entities(entities)
    entities = resolve_overlaps(entities)

    # Keep entity text synchronized with original tweet.
    for e in entities:
        e["text"] = text[e["start"]:e["end"]]
        e["normalized_text"] = normalize_entity_text(
            e["text"], e["label"]
        )

    # Relations
    request_entities = [
        e for e in entities if e["label"] == "REQUEST"
    ]
    resource_entities = [
        e for e in entities if e["label"] == "RESOURCE"
    ]

    links = link_request_to_resources(
        text,
        request_entities,
        resource_entities
    )

    return {
        "text": text,
        "entities": entities,
        "request_resource_links": links
    }


In [ ]:
# ============================================================
# 16. TEST HYBRID PIPELINE ON HAND-WRITTEN EXAMPLES
# ============================================================

EXAMPLES = [
    "Flooding in Mumbai. 20 people injured and families need drinking water.",
    "Bridge collapsed after the earthquake. 5 people are trapped inside.",
    "Red Cross is providing medicine to displaced residents.",
    "Urgently need blankets and food in Houston.",
    "Fire destroyed several houses and roads near the airport."
]

for i, text in enumerate(EXAMPLES, 1):
    result = extract_hybrid(text)

    print(f"\n{'='*80}")
    print(f"EXAMPLE {i}")
    print(text)
    print("-" * 80)

    for entity in result["entities"]:
        print(
            f"{entity['label']:18s} | "
            f"{entity['text']!r:30s} | "
            f"{entity['score']:.2f} | "
            f"{entity['source']}"
        )


# 17. Convert entities into the structured incident representation

The structured record is the important output of Phase 4.

The raw entity list is retained for traceability, while the grouped fields provide the representation consumed by later phases.


In [ ]:
# ============================================================
# 17. STRUCTURED INCIDENT JSON
# ============================================================

FIELD_MAP = {
    "LOCATION": "location",
    "CASUALTY": "casualties",
    "DISPLACED": "displaced",
    "REQUEST": "requests",
    "RESOURCE": "resources",
    "RESCUE": "rescue",
    "DISASTER_TYPE": "disaster_type",
    "ORGANIZATION": "organizations",
    "PERSON": "persons",
    "NUMBER": "numbers",
    "INFRASTRUCTURE": "infrastructure"
}

def build_structured_record(result):
    text = result["text"]
    entities = result["entities"]

    record = {
        "text": text,
        "location": [],
        "casualties": [],
        "displaced": [],
        "requests": [],
        "resources": [],
        "rescue": [],
        "disaster_type": [],
        "organizations": [],
        "persons": [],
        "numbers": [],
        "infrastructure": [],
        "entities": entities,
        "request_resource_links": result["request_resource_links"]
    }

    for entity in entities:
        label = entity["label"]

        if label not in FIELD_MAP:
            continue

        field = FIELD_MAP[label]

        if entity["text"] not in record[field]:
            record[field].append(entity["text"])

    # Disaster type can be represented as a single canonical value
    # when exactly one type is detected.
    if len(record["disaster_type"]) == 1:
        record["disaster_type"] = record["disaster_type"][0]

    return record


example_record = build_structured_record(
    extract_hybrid(EXAMPLES[0])
)

print(json.dumps(example_record, indent=2, ensure_ascii=False))


# 18. Run on real HumAID tweets

This is the first real end-to-end pass.

For a development run, `DEMO_LIMIT=2000` keeps iteration manageable. Set it to `None` when you are ready to process the selected split completely.


In [ ]:
# ============================================================
# 18. RUN HYBRID EXTRACTION ON HUM AID
# ============================================================

results = []

for text in tqdm(
    df[TEXT_COLUMN].tolist(),
    desc="Hybrid extraction"
):
    result = extract_hybrid(text)
    structured = build_structured_record(result)
    results.append(structured)

results_df = pd.DataFrame(results)

print("Processed:", len(results_df))
display(results_df.head(5))


In [ ]:
# ============================================================
# 19. ENTITY STATISTICS
# ============================================================

entity_counter = defaultdict(int)
source_counter = defaultdict(int)

for result in results:
    for e in result["entities"]:
        entity_counter[e["label"]] += 1
        source_counter[e["source"]] += 1

entity_stats = (
    pd.DataFrame(
        sorted(entity_counter.items(), key=lambda x: x[1], reverse=True),
        columns=["entity_type", "count"]
    )
)

source_stats = (
    pd.DataFrame(
        sorted(source_counter.items(), key=lambda x: x[1], reverse=True),
        columns=["source", "count"]
    )
)

print("Entity counts:")
display(entity_stats)

print("Extraction-source counts:")
display(source_stats)


In [ ]:
# ============================================================
# 20. INSPECT INTERESTING EXTRACTIONS
# ============================================================

def show_examples(label, n=10):
    rows = []

    for result in results:
        matches = [
            e for e in result["entities"]
            if e["label"] == label
        ]

        if matches:
            rows.append({
                "text": result["text"],
                "entities": "; ".join(
                    f"{e['text']} ({e['score']:.2f})"
                    for e in matches
                )
            })

        if len(rows) >= n:
            break

    return pd.DataFrame(rows)

for label in [
    "LOCATION",
    "CASUALTY",
    "REQUEST",
    "RESOURCE",
    "RESCUE",
    "DISPLACED",
    "INFRASTRUCTURE",
    "DISASTER_TYPE"
]:
    print("\n", "=" * 80)
    print(label)
    display(show_examples(label, n=5))


# 21. Gold-standard annotation sample

We do **not** claim the rule outputs are ground truth.

Instead, create a small manually reviewed sample. The easiest workflow is:

1. Export candidate tweets.
2. Annotate them manually.
3. Store the gold entities using character offsets.
4. Compare predicted spans/labels against the gold annotations.

Recommended annotation coverage should deliberately include difficult cases:

- casualty numbers
- implicit casualty statements
- resource requests
- rescue situations
- displacement
- infrastructure damage
- multiple locations
- organizations/persons
- ambiguous words
- tweets containing several entity types
- tweets with no extractable entities


In [ ]:
# ============================================================
# 22. CREATE ANNOTATION TEMPLATE
# ============================================================

ANNOTATION_COLUMNS = [
    "annotation_id",
    "text",
    "entities_json"
]

# Select a balanced-ish sample from the processed data.
# We include tweets with entities plus some tweets with no entities.
has_entities = results_df["entities"].apply(len).gt(0)

positive = results_df[has_entities]
negative = results_df[~has_entities]

n_positive = min(100, len(positive))
n_negative = min(50, len(negative))

annotation_sample = pd.concat([
    positive.sample(n=n_positive, random_state=SEED) if n_positive else positive,
    negative.sample(n=n_negative, random_state=SEED) if n_negative else negative
], ignore_index=True)

annotation_sample = annotation_sample.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

annotation_template = pd.DataFrame({
    "annotation_id": np.arange(len(annotation_sample)),
    "text": annotation_sample["text"].values,
    "entities_json": [
        "[]"
        for _ in range(len(annotation_sample))
    ]
})

annotation_path = OUTPUT_DIR / "gold_annotation_template.csv"
annotation_template.to_csv(annotation_path, index=False)

print("Annotation template saved to:", annotation_path)
display(annotation_template.head(10))


## Gold annotation format

For each row, replace `entities_json` with a JSON list like:

```json
[
  {
    "text": "Mumbai",
    "start": 12,
    "end": 18,
    "label": "LOCATION"
  },
  {
    "text": "20 people injured",
    "start": 21,
    "end": 38,
    "label": "CASUALTY"
  },
  {
    "text": "drinking water",
    "start": 55,
    "end": 69,
    "label": "RESOURCE"
  }
]
```

Use character offsets from the original tweet. This makes evaluation deterministic and reproducible.


In [ ]:
# ============================================================
# 23. GOLD FILE LOADER
# ============================================================

GOLD_FILE = OUTPUT_DIR / "gold_annotations.csv"

def load_gold_annotations(path):
    if not path.exists():
        print(
            "Gold annotation file not found yet. "
            "Complete manual annotation first:"
        )
        print(path)
        return None

    gold = pd.read_csv(path)

    required = {"annotation_id", "text", "entities_json"}
    missing = required - set(gold.columns)

    if missing:
        raise ValueError(
            f"Gold file is missing columns: {sorted(missing)}"
        )

    parsed = []

    for raw in gold["entities_json"].fillna("[]"):
        try:
            entities = json.loads(raw)
            if not isinstance(entities, list):
                raise ValueError
            parsed.append(entities)
        except Exception:
            parsed.append([])

    gold["gold_entities"] = parsed

    return gold


gold_df = load_gold_annotations(GOLD_FILE)

if gold_df is not None:
    print("Gold rows:", len(gold_df))
    display(gold_df.head())


In [ ]:
# ============================================================
# 24. ENTITY-LEVEL EVALUATION
# ============================================================

def entity_key(entity):
    return (
        int(entity["start"]),
        int(entity["end"]),
        str(entity["label"])
    )

def evaluate_entity_predictions(pred_records, gold_df):
    pred_by_text = defaultdict(list)

    for record in pred_records:
        pred_by_text[record["text"]].extend(record["entities"])

    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    evaluated = 0

    for _, row in gold_df.iterrows():
        text = str(row["text"])
        gold_entities = row["gold_entities"]

        pred_entities = pred_by_text.get(text, [])

        gold_keys = {
            entity_key(e)
            for e in gold_entities
            if all(k in e for k in ["start", "end", "label"])
        }

        pred_keys = {
            entity_key(e)
            for e in pred_entities
            if all(k in e for k in ["start", "end", "label"])
        }

        for key in pred_keys & gold_keys:
            tp[key[2]] += 1

        for key in pred_keys - gold_keys:
            fp[key[2]] += 1

        for key in gold_keys - pred_keys:
            fn[key[2]] += 1

        evaluated += 1

    labels = sorted(set(tp) | set(fp) | set(fn))

    rows = []

    total_tp = total_fp = total_fn = 0

    for label in labels:
        t = tp[label]
        f_p = fp[label]
        f_n = fn[label]

        precision = t / (t + f_p) if (t + f_p) else 0.0
        recall = t / (t + f_n) if (t + f_n) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        rows.append({
            "label": label,
            "TP": t,
            "FP": f_p,
            "FN": f_n,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

        total_tp += t
        total_fp += f_p
        total_fn += f_n

    micro_precision = (
        total_tp / (total_tp + total_fp)
        if (total_tp + total_fp) else 0.0
    )
    micro_recall = (
        total_tp / (total_tp + total_fn)
        if (total_tp + total_fn) else 0.0
    )
    micro_f1 = (
        2 * micro_precision * micro_recall /
        (micro_precision + micro_recall)
        if (micro_precision + micro_recall) else 0.0
    )

    report = pd.DataFrame(rows)

    summary = {
        "evaluated_tweets": evaluated,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1
    }

    return report, summary


if gold_df is not None:
    gold_report, gold_summary = evaluate_entity_predictions(
        results,
        gold_df
    )

    print("Gold-standard evaluation:")
    print(json.dumps(gold_summary, indent=2))
    display(gold_report.sort_values("f1", ascending=False))
else:
    print("Evaluation skipped until gold_annotations.csv is created.")


# 25. Error analysis

For each gold sample, inspect:

- false positives
- false negatives
- wrong entity type
- boundary errors
- ambiguous requests
- implicit casualties
- context-dependent resource mentions

This is where the rule layer should be improved instead of blindly increasing rule coverage.


In [ ]:
# ============================================================
# 25. ERROR ANALYSIS
# ============================================================

def error_analysis(pred_records, gold_df):
    pred_by_text = defaultdict(list)

    for record in pred_records:
        pred_by_text[record["text"]].extend(record["entities"])

    rows = []

    for _, row in gold_df.iterrows():
        text = str(row["text"])
        gold_entities = row["gold_entities"]
        pred_entities = pred_by_text.get(text, [])

        gold_keys = {
            entity_key(e): e for e in gold_entities
            if all(k in e for k in ["start", "end", "label"])
        }

        pred_keys = {
            entity_key(e): e for e in pred_entities
            if all(k in e for k in ["start", "end", "label"])
        }

        for key in pred_keys.keys() - gold_keys.keys():
            rows.append({
                "error_type": "FALSE_POSITIVE",
                "text": text,
                "span": pred_keys[key].get("text", ""),
                "label": pred_keys[key].get("label", "")
            })

        for key in gold_keys.keys() - pred_keys.keys():
            start, end, label = key
            rows.append({
                "error_type": "FALSE_NEGATIVE",
                "text": text,
                "span": text[start:end],
                "label": label
            })

    return pd.DataFrame(rows)


if gold_df is not None:
    errors_df = error_analysis(results, gold_df)

    print("Errors:", len(errors_df))
    display(errors_df.head(30))
else:
    errors_df = pd.DataFrame()
    print("Error analysis will run after gold annotations are available.")


In [ ]:
# ============================================================
# 26. SAVE ALL OUTPUTS
# ============================================================

# Flat entity table
entity_rows = []

for idx, result in enumerate(results):
    for e in result["entities"]:
        entity_rows.append({
            "row_id": idx,
            "text": result["text"],
            **e
        })

entities_df = pd.DataFrame(entity_rows)

entities_path = OUTPUT_DIR / "phase4_3_entities.csv"
structured_path = OUTPUT_DIR / "phase4_3_structured_incidents.jsonl"
summary_path = OUTPUT_DIR / "phase4_3_summary.csv"

entities_df.to_csv(entities_path, index=False)

with open(structured_path, "w", encoding="utf-8") as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

summary_df = pd.DataFrame({
    "metric": [
        "processed_tweets",
        "tweets_with_entities",
        "total_entities",
        "output_directory"
    ],
    "value": [
        len(results),
        sum(len(r["entities"]) > 0 for r in results),
        sum(len(r["entities"]) for r in results),
        str(OUTPUT_DIR)
    ]
})

summary_df.to_csv(summary_path, index=False)

print("Saved:")
print(" -", entities_path)
print(" -", structured_path)
print(" -", summary_path)

if gold_df is not None:
    gold_report.to_csv(
        OUTPUT_DIR / "gold_entity_evaluation.csv",
        index=False
    )

    with open(
        OUTPUT_DIR / "gold_evaluation_summary.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(gold_summary, f, indent=2)


# 27. Final Phase 4.3 checklist

Before declaring this stage complete:

- [ ] General NER runs successfully.
- [ ] Actual HumAID columns/splits were inspected.
- [ ] Disaster-specific gazetteers are documented.
- [ ] Casualty extraction works on reviewed examples.
- [ ] Request/resource extraction works on reviewed examples.
- [ ] Rescue extraction works on reviewed examples.
- [ ] Displacement extraction works on reviewed examples.
- [ ] Disaster-type extraction works.
- [ ] Infrastructure extraction works.
- [ ] Entity normalization and merging work.
- [ ] Conflict resolution is inspected.
- [ ] Structured incident JSON is generated.
- [ ] Real HumAID tweets have been processed.
- [ ] A gold-standard sample has been manually annotated.
- [ ] Entity-level Precision / Recall / F1 are reported.
- [ ] False positives and false negatives are reviewed.
- [ ] Final structured dataset is exported.

### Important interpretation

Do **not** report the general NER model's CoNLL score as the performance of our disaster information extractor. The model card's reported scores are for CoNLL-2003, while our project has a different domain and entity schema. citeturn0search2

Our project's defensible NER result is the evaluation against our own manually annotated gold-standard disaster sample.


# Phase 4.3 output

At the end of this notebook we have:

```text
Raw HumAID Tweet
        ↓
General NER
        ↓
Disaster-specific extraction
        ↓
Normalization
        ↓
Conflict resolution
        ↓
Structured incident
        ↓
Gold-standard evaluation
        ↓
Exported JSONL / CSV
```

This structured incident layer is the input contract for **Phase 5 — Intelligence**, where we will build severity, urgency, incident clustering/deduplication, high-risk zones, and resource estimation.
